# EDA da Mega-Sena

Este caderno conta a história dos **concursos** da Mega-Sena a partir do dado processado real
(**3069 concursos** do arquivo `data/raw/Mega-Sena.csv`, processados com `src/data`).
Vamos olhar **frequência** por **dezena**, **atraso** por dezena, perfil de **pares/ímpares**,
**soma** por concurso e a **coocorrência** entre dezenas.

**Aviso importante:** loteria é aleatória. Este caderno é descritivo e educacional —
mostra o que já saiu, mas **não promete acerto, não prevê o próximo concurso
e não recomenda aposta**.

In [ ]:
# Setup: resolve repo paths and load the real processed concursos.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "config.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from src.data import eda, load, process

raw_candidates = [Path("data/raw/Mega-Sena.csv"), Path("../data/raw/Mega-Sena.csv")]
raw_path = next(p for p in raw_candidates if p.exists())
raw = load.load_raw_csv(raw_path)
df = process.process_concursos(raw)
print(f"concursos: {len(df)} | columns: {list(df.columns)}")
df.head(3)

## Frequência por dezena

Quantas vezes cada dezena de 1 a 60 apareceu nos 3069 concursos.
Se o sorteio fosse perfeitamente uniforme, cada dezena sairia cerca de 307 vezes
(3069 × 6 / 60). Vamos ver o quanto a realidade se aproxima disso.

In [ ]:
# Frequency per dezena bar chart.
freq = process.compute_frequencia(df)
top5 = freq.sort_values("frequencia", ascending=False).head(5)
bottom5 = freq.sort_values("frequencia").head(5)
print("most frequent:")
print(top5.to_string(index=False))
print("least frequent:")
print(bottom5.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.bar(freq["dezena"], freq["frequencia"], color="steelblue")
ax.axhline(
    freq["frequencia"].mean(), color="red", linestyle="--", linewidth=1, label="mean"
)
ax.set_title("Frequência por dezena (3069 concursos)")
ax.set_xlabel("Dezena")
ax.set_ylabel("Concursos com a dezena")
ax.set_xticks(list(range(1, 61)))
ax.tick_params(axis="x", rotation=90, labelsize=8)
ax.legend()
fig.tight_layout()

### O que a frequência conta

As dezenas mais sorteadas (10, 53, 5) passam de 330 aparições e as menos sorteadas
(26, 21, 55) ficam perto de 250 — uma diferença visível, mas pequena diante de 3069 concursos.
Isso é o comportamento esperado de um sorteio aleatório com milhares de repetições:
alguma variação aparece, sem formar um padrão aproveitável. Nada aqui indica qual dezena
"merece" aposta no próximo concurso.

## Atraso por dezena

Há quantos concursos cada dezena não sai. Atraso 0 significa que ela saiu no último concurso;
valores altos são as dezenas que estão há mais tempo sem aparecer.

In [ ]:
# Delay per dezena bar chart.
delay = process.compute_atraso(df)
most_due = delay.sort_values("atraso", ascending=False).head(8)
print("longest delay:")
print(most_due.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.bar(delay["dezena"], delay["atraso"], color="darkorange")
ax.set_title("Atraso por dezena (concursos desde a última aparição)")
ax.set_xlabel("Dezena")
ax.set_ylabel("Concursos de atraso")
ax.set_xticks(list(range(1, 61)))
ax.tick_params(axis="x", rotation=90, labelsize=8)
fig.tight_layout()

### O que o atraso conta

No fechamento do histórico, as dezenas com maior atraso (como 34 e 45, perto de 40 concursos)
são exceção: a maioria voltou a sair há poucos concursos. Atraso alto é curiosidade histórica,
não sinal de que a dezena esteja "devendo" — cada concurso sorteia as 6 dezenas do zero,
sem memória do anterior.

## Pares e ímpares por concurso

Quantos concursos saíram com 0, 1, 2, ... 6 dezenas pares. O jogo típico mistura os dois lados;
os extremos (tudo par ou tudo ímpar) costumam ser raros.

In [ ]:
# Parity distribution bar chart.
parity = eda.compute_parity_distribution(df)
print(parity.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(parity["n_pares"], parity["concursos"], color="teal")
ax.set_title("Concursos por quantidade de pares no jogo")
ax.set_xlabel("Pares no jogo (0 a 6)")
ax.set_ylabel("Concursos")
ax.set_xticks(list(range(7)))
fig.tight_layout()

### O que pares/ímpares contam

O pico está em **3 pares e 3 ímpares** (944 concursos), com 2 e 4 pares logo atrás
(737 e 747). Jogos com 0, 1, 5 ou 6 pares somam menos de 600 concursos juntos.
Ou seja: o sorteio costuma equilibrar os dois lados, e os extremos são minoria —
um retrato do passado, não uma regra para o próximo jogo.

## Soma por concurso

A soma das 6 dezenas de cada concurso resume se o jogo sorteado foi "baixo" (dezenas pequenas)
ou "alto" (dezenas grandes). Olhamos a distribuição geral e a série ao longo dos concursos.

In [ ]:
# Soma distribution and soma-over-time charts.
summary = eda.compute_soma_summary(df)
print(
    f"count={summary['count']} min={summary['min']} "
    f"max={summary['max']} mean={summary['mean']:.1f} "
    f"median={summary['median']:.1f}"
)

fig, (top_ax, bottom_ax) = plt.subplots(2, 1, figsize=(12, 8))
top_ax.hist(df["soma"], bins=30, color="slateblue", edgecolor="white")
top_ax.axvline(
    df["soma"].mean(), color="red", linestyle="--", linewidth=1, label="mean"
)
top_ax.set_title("Distribuição da soma por concurso")
top_ax.set_xlabel("Soma das 6 dezenas")
top_ax.set_ylabel("Concursos")
top_ax.legend()
bottom_ax.plot(df["concurso"], df["soma"], linewidth=0.5, color="slateblue", alpha=0.7)
bottom_ax.axhline(
    df["soma"].mean(), color="red", linestyle="--", linewidth=1, label="mean"
)
bottom_ax.set_title("Soma por concurso ao longo do histórico")
bottom_ax.set_xlabel("Concurso")
bottom_ax.set_ylabel("Soma")
bottom_ax.legend()
fig.tight_layout()

### O que a soma conta

A soma típica fica perto de **183 (média) e 184 (mediana)**, com a maior parte dos concursos
entre 150 e 210 e extremos raros (mínimo 63, máximo 331). A série no tempo não mostra tendência
de alta ou queda — só a nuvem estável de um sorteio que se repete. Serve para descrever
o "jogo médio", não para escolher o próximo.

## Mapa de calor: quais dezenas saem juntas

Cada célula conta em quantos concursos o par de dezenas saiu no mesmo jogo.
A diagonal é a frequência de cada dezena; fora dela, valores altos indicariam
duplas que costumam aparecer juntas.

In [ ]:
# Co-occurrence heatmap (tested helper).
matrix = eda.compute_cooccurrence_matrix(df)
off_diag = matrix.where(~np.eye(len(matrix), dtype=bool))
print(
    f"matrix shape: {matrix.shape} | max pair co-occurrence: {int(np.nanmax(off_diag.values))}"
)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    matrix,
    cmap="viridis",
    square=True,
    ax=ax,
    xticklabels=5,
    yticklabels=5,
    cbar_kws={"label": "Concursos juntos"},
)
ax.set_title("Coocorrência de dezenas (pares no mesmo concurso)")
ax.set_xlabel("Dezena")
ax.set_ylabel("Dezena")
fig.tight_layout()

### O que o mapa de calor conta

O mapa é quase uniforme, sem blocos fortes nem duplas dominantes: os pares se distribuem
de forma difusa, como esperado quando 6 dezenas são sorteadas ao acaso entre 60.
Se houvesse estrutura (por exemplo, sempre os mesmos pares juntos), ela apareceria aqui
como manchas claras — não é o caso.

## Resumo

- **Frequência:** variação modesta em torno do esperado (~307 por dezena); nada sugere dezena favorecida.
- **Atraso:** poucas dezenas passam de 30 concursos sem sair; atraso não é promessa de retorno.
- **Pares/ímpares:** o jogo típico tem 3 pares e 3 ímpares; extremos são raros.
- **Soma:** centro em ~183, sem tendência no tempo.
- **Coocorrência:** sem duplas dominantes, compatível com sorteio aleatório.

Em conjunto, os gráficos descrevem um histórico estável e barulhento. Este caderno **não gera
palpite, não ranqueia dezenas para aposta e não afirma que qualquer padrão prevê o futuro** —
é apenas o retrato honesto de 3069 concursos.